# Alpaca Trading & Broker API — Learn the APIs

**Purpose**: Learn Alpaca APIs (Paper Trading & Broker API) for Project Yagnum
**Cost**: $0 — all simulated Paper Trading & Sandbox
**Time**: ~10-15 min to run all cells

| Step | What You Learn | API Used |
|------|---------------|----------|
| 0 | Setup & Dependencies | pip, requests, dotenv |
| 1 | Load Environment Credentials | .env config |
| 2 | Check Market Clock & Status | `/v2/clock` |
| 3 | Fetch Account Details & Balances | `/v2/account` |
| 4 | Explore Assets & Market Data | `/v2/assets`, `/v2/stocks/quotes` |
| 5 | Broker API: Create & Manage Sub-Accounts | `/v1/accounts` (Broker Sandbox) |
| 6 | Execute Simulated Paper Orders | `/v2/orders`, `/v2/positions` |


## Step 0 — Install & Import Dependencies

We use `requests` for REST calls, `python-dotenv` for configuration, and `pandas` for data display.

In [ ]:
import os
import json
import base64
import requests
import pandas as pd
from dotenv import load_dotenv

# Load variables from the root .env file
load_dotenv(dotenv_path="../.env")

print("All libraries loaded successfully!")

## Step 1 — Configure Credentials

Alpaca supports two API types:
1. **Individual Trading API (Paper Sandbox)**: Uses `APCA-API-KEY-ID` (`PK...`) & `APCA-API-SECRET-KEY`
2. **Broker API (Sandbox)**: Uses HTTP Basic Auth (`ALPACA_BROKER_ID` & `ALPACA_BROKER_SECRET`)

In [ ]:
# Base URLs
PAPER_BASE_URL = "https://paper-api.alpaca.markets"
DATA_BASE_URL = "https://data.alpaca.markets"
BROKER_BASE_URL = "https://broker-api.sandbox.alpaca.markets"

# 1. Individual Trading API Credentials (if you have PK... keys)
ALPACA_API_KEY = os.getenv("ALPACA_API_KEY", os.getenv("APCA_API_KEY_ID", ""))
ALPACA_API_SECRET = os.getenv("ALPACA_API_SECRET", os.getenv("APCA_API_SECRET_KEY", ""))

# 2. Broker API Credentials (for Broker Accounts / multi-tenant)
BROKER_ID = os.getenv("ALPACA_BROKER_ID", "")
BROKER_SECRET = os.getenv("ALPACA_BROKER_SECRET", "")

print("Credentials Configured:")
print(f" - Trading API Key: {'Set (' + ALPACA_API_KEY[:6] + '...)' if ALPACA_API_KEY else 'Not Set'}")
print(f" - Broker Client ID: {'Set (' + BROKER_ID[:6] + '...)' if BROKER_ID else 'Not Set'}")

## Step 2 — Check Market Clock & Status

Check whether the US stock market is currently open or closed, and when the next session starts.

In [ ]:
def check_market_clock():
    # Standard Paper Trading endpoint (can also be called on Broker endpoint)
    headers = {
        "APCA-API-KEY-ID": ALPACA_API_KEY,
        "APCA-API-SECRET-KEY": ALPACA_API_SECRET
    }
    
    url = f"{PAPER_BASE_URL}/v2/clock"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        clock = response.json()
        print(f"Timestamp: {clock.get('timestamp')}")
        print(f"Market Open?: {clock.get('is_open')}")
        print(f"Next Open:    {clock.get('next_open')}")
        print(f"Next Close:   {clock.get('next_close')}")
    else:
        print(f"Request failed (Status {response.status_code}):")
        print(response.text)

check_market_clock()

## Step 3 — Get Account Details & Buying Power

Inspect cash, buying power, equity, and margin multiplier in your simulated account.

In [ ]:
def get_account_summary():
    headers = {
        "APCA-API-KEY-ID": ALPACA_API_KEY,
        "APCA-API-SECRET-KEY": ALPACA_API_SECRET
    }
    url = f"{PAPER_BASE_URL}/v2/account"
    res = requests.get(url, headers=headers)
    
    if res.status_code == 200:
        acc = res.json()
        df = pd.DataFrame([{
            "Account Number": acc.get("account_number"),
            "Status": acc.get("status"),
            "Currency": acc.get("currency"),
            "Cash": f"${float(acc.get('cash', 0)):,.2f}",
            "Buying Power": f"${float(acc.get('buying_power', 0)):,.2f}",
            "Portfolio Value": f"${float(acc.get('portfolio_value', 0)):,.2f}",
            "Multiplier": acc.get("multiplier")
        }])
        return df
    else:
        print(f"Error {res.status_code}: {res.text}")
        return None

get_account_summary()

## Step 4 — Search Assets & Market Data

Query stock data, asset status, and current bid/ask prices.

In [ ]:
def inspect_asset(symbol="AAPL"):
    headers = {
        "APCA-API-KEY-ID": ALPACA_API_KEY,
        "APCA-API-SECRET-KEY": ALPACA_API_SECRET
    }
    
    # 1. Asset Metadata
    asset_res = requests.get(f"{PAPER_BASE_URL}/v2/assets/{symbol}", headers=headers)
    if asset_res.status_code == 200:
        asset = asset_res.json()
        print(f"Asset: {asset.get('name')} ({asset.get('symbol')}) - Exchange: {asset.get('exchange')}")
        print(f"Tradable: {asset.get('tradable')} | Marginable: {asset.get('marginable')} | Fractionable: {asset.get('fractionable')}")
    
    # 2. Latest Quote from Market Data API
    quote_res = requests.get(f"{DATA_BASE_URL}/v2/stocks/{symbol}/quotes/latest", headers=headers)
    if quote_res.status_code == 200:
        q = quote_res.json().get("quote", {})
        print(f"\nLatest Quote for {symbol}:")
        print(f"  Bid: ${q.get('bp')} (size: {q.get('bs')})")
        print(f"  Ask: ${q.get('ap')} (size: {q.get('as')})")
        print(f"  Time: {q.get('t')}")

inspect_asset("AAPL")

## Step 5 — Broker API: Creating Sub-Accounts

If you are using the **Alpaca Broker API**, this demonstrates how to create a new customer account programmatically in the Broker Sandbox.

In [ ]:
def create_broker_subaccount(first_name="Alice", last_name="Smith", email="alice@example.com"):
    """
    Creates a simulated individual brokerage sub-account in Alpaca Broker Sandbox.
    Uses HTTP Basic Authentication.
    """
    auth = (BROKER_ID, BROKER_SECRET)
    url = f"{BROKER_BASE_URL}/v1/accounts"
    
    # Sample payload required by Alpaca Broker API
    payload = {
        "contact": {
            "email_address": email,
            "phone_number": "555-555-5555",
            "street_address": ["123 Main St"],
            "city": "San Mateo",
            "state": "CA",
            "postal_code": "94401",
            "country": "USA"
        },
        "identity": {
            "given_name": first_name,
            "family_name": last_name,
            "date_of_birth": "1990-01-01",
            "tax_id": "666-55-4321",
            "tax_id_type": "USA_SSN",
            "country_of_citizenship": "USA",
            "country_of_birth": "USA",
            "country_of_tax_residence": "USA",
            "funding_source": ["employment_income"]
        },
        "disclosures": {
            "is_control_person": False,
            "is_affiliated_exchange_or_finra": False,
            "is_politically_exposed": False,
            "immediate_family_exposed": False
        },
        "agreements": [
            {
                "agreement": "margin_agreement",
                "signed_at": "2026-08-22T00:00:00Z",
                "ip_address": "127.0.0.1"
            },
            {
                "agreement": "account_agreement",
                "signed_at": "2026-08-22T00:00:00Z",
                "ip_address": "127.0.0.1"
            },
            {
                "agreement": "customer_agreement",
                "signed_at": "2026-08-22T00:00:00Z",
                "ip_address": "127.0.0.1"
            }
        ]
    }
    
    res = requests.post(url, auth=auth, json=payload)
    print(f"Status Code: {res.status_code}")
    try:
        print(json.dumps(res.json(), indent=2))
    except Exception:
        print(res.text)

# Uncomment to test subaccount creation in Broker Sandbox:
# create_broker_subaccount()

## Step 6 — Place a Paper Trade Order

Submit a market order to buy 1 share of a stock and check active positions.

In [ ]:
def place_market_order(symbol="AAPL", qty=1, side="buy"):
    headers = {
        "APCA-API-KEY-ID": ALPACA_API_KEY,
        "APCA-API-SECRET-KEY": ALPACA_API_SECRET,
        "Content-Type": "application/json"
    }
    
    order_payload = {
        "symbol": symbol,
        "qty": qty,
        "side": side,
        "type": "market",
        "time_in_force": "day"
    }
    
    url = f"{PAPER_BASE_URL}/v2/orders"
    res = requests.post(url, headers=headers, json=order_payload)
    
    if res.status_code in (200, 201):
        order = res.json()
        print(f"Order Submitted Successfully!")
        print(f"  Order ID: {order.get('id')}")
        print(f"  Symbol:   {order.get('symbol')}")
        print(f"  Qty:      {order.get('qty')}")
        print(f"  Side:     {order.get('side')}")
        print(f"  Status:   {order.get('status')}")
    else:
        print(f"Order Failed ({res.status_code}): {res.text}")

# Uncomment to test placing an order in Paper Trading:
# place_market_order(symbol="AAPL", qty=1, side="buy")